# canvas-ctx: spatial habitat inference from H&E

Runs the full pipeline in Google Colab. The same scripts run locally in
VS Code; nothing here is Colab-specific except the setup cell and the
GPU detection.

**Runtime:** `Runtime > Change runtime type > T4 GPU` is recommended. Patch
encoding is roughly 20x faster on a T4 than on CPU. Everything else
(neighbourhood discovery, spatial features, survival models) is CPU-bound
and unaffected.

**Storage:** the Orion paired cohort is about 1.7 GB per specimen once the
multiplex images are skipped. Mount Drive in section 1 if you want the
downloads and cached embeddings to survive a runtime restart.

## 1. Environment

In [ ]:
# Optional: persist data and results across runtime restarts.
USE_DRIVE = False

import os, pathlib

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = pathlib.Path('/content/drive/MyDrive/canvas-ctx')
else:
    WORKDIR = pathlib.Path('/content/canvas-ctx')

WORKDIR.parent.mkdir(parents=True, exist_ok=True)
print('working directory:', WORKDIR)

In [ ]:
REPO_URL = 'https://github.com/swatian1989/canvas-ctx.git'

import subprocess, sys, pathlib

if not (WORKDIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)

%cd {WORKDIR}
!git pull --ff-only || true

In [ ]:
# Colab ships torch, numpy, pandas and matplotlib. Install the rest.
!pip install -q lifelines scikit-survival python-igraph tifffile imagecodecs \
               zarr timm transformers python-docx tabulate openpyxl \
               opencv-python-headless
!pip install -q -e .

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('No GPU. Encoding will be slow; consider Runtime > Change runtime type.')

In [ ]:
# Gated encoders (UNI, MUSK) need an access token. Phikon is ungated and is
# the default, so this cell is optional. The token is read from the
# environment and never written to disk.
import os, getpass

SET_TOKEN = False
if SET_TOKEN:
    os.environ['HF_TOKEN'] = getpass.getpass('HuggingFace token: ')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
!python -m pytest tests/ -q

## 2. Stage 1: cellular neighbourhood discovery

Downloads the Schürch colorectal CODEX table (223 MB), converts pixel
coordinates to microns at the resolution documented for that collection,
and discovers neighbourhoods with a 40 µm radius.

The unit conversion is checked against the data: median nearest-neighbour
spacing should land near 7 µm. If it does not, stop and investigate rather
than proceeding, because a wrong scale still produces clean-looking
clusters.

In [ ]:
import pathlib, urllib.request

dest = pathlib.Path('data/raw/CRC_clusters_neighborhoods_markers.csv')
dest.parent.mkdir(parents=True, exist_ok=True)
URL = ('https://data.mendeley.com/public-files/datasets/mpjzbtfgfr/files/'
       'c24351b3-76d7-444f-9edf-0246356b0c78/file_downloaded')

if not dest.exists():
    print('downloading 223 MB ...')
    urllib.request.urlretrieve(URL, dest)
print(f'{dest} : {dest.stat().st_size/1e6:.0f} MB')

In [ ]:
!python scripts/prepare_schurch.py
!python scripts/run_stage1_cn.py --config config/crc_train_brca_apply.yaml \
    --cells data/interim/schurch_crc_cells.parquet --outdir data/processed

In [ ]:
# External validation: the source table ships the original authors' own
# neighbourhood labels, so the rediscovered ones can be scored against them.
!python scripts/validate_cn_vs_published.py

## 3. Stage 2: paired cohort

Orion images 18-plex immunofluorescence and H&E from the same section, so
the expected transform between them is the identity. That is verified, not
assumed: the check cross-correlates the H&E tissue mask against the cell
density map and reports the offset as a residual in microns.

Only the registered H&E and the single-cell table are downloaded. The
19-channel multiplex image for each specimen is 44 to 147 GB and nothing
downstream reads it.

In [ ]:
# Start with three specimens to keep the download small. Twelve gives the
# 8/2/2 patient-level split the protocol specifies.
SPECIMENS = ['CRC01', 'CRC02', 'CRC03']

!python scripts/download_orion.py --specimens {' '.join(SPECIMENS)}

In [ ]:
!python scripts/run_orion_registration_qc.py --level 3

import json
qc = json.load(open('results/orion_registration_qc.json'))
print(json.dumps(qc, indent=2))
assert qc['verdict'] == 'PASS', 'registration exceeds the 5 um threshold; do not proceed'

In [ ]:
from IPython.display import Image, display
display(Image('figures/F6_registration_qc.png'))

In [ ]:
# One shared habitat taxonomy across every specimen, then patch labels
# under the CANVAS purity rules.
!python scripts/run_orion_cohort.py

In [ ]:
# Real H&E patches grouped by dominant gated lineage. This doubles as a
# visual audit of the marker gating: tumour epithelium and smooth muscle
# are checkable by eye.
!python scripts/extract_orion_patches.py
display(Image('figures/F7_patch_labels.png'))

## 4. Stage 3: cache patch embeddings

The slow step, and the reason a GPU helps. Embeddings are cached to parquet
so the classifier head can be retrained without re-encoding. Encoding is
resumable per specimen.

In [ ]:
BATCH = 64 if DEVICE == 'cuda' else 16
!python scripts/encode_orion_patches.py --encoder phikon --batch-size {BATCH}

## 5. Method 1 versus Method 2

The controlled ablation. `none` is CANVAS exactly; the other three differ
only in how neighbourhood evidence reaches the same classification head.

Six seeds minimum: a paired signed-rank test cannot go below p = 0.0312
with six pairs, and with three the floor is 0.25. Report macro-F1 and
Cohen's kappa, never accuracy alone.

In [ ]:
!python scripts/run_final_benchmark.py \
    --embeddings data/interim/orion_embeddings \
    --modes none graph grid2d grid3d \
    --seeds 1 2 3 4 5 6 --epochs 30 --window 7 \
    --outdir results/real_benchmark

In [ ]:
import pandas as pd

res = pd.read_csv('results/real_benchmark/final_benchmark.csv')
summary = (res[res.metric.isin(['macro_f1', 'cohen_kappa'])]
             .groupby(['metric', 'mode'])['value']
             .agg(['mean', 'std', 'count'])
             .round(4))
print(summary.to_string())

## 6. Stages 5 and 6: spatial features and clinical modelling

The 262-feature engine and the survival chain run on CPU in minutes. Run
them against simulated habitat maps first: on a random outcome the
false-discovery-corrected Cox results should be null, and anything
significant indicates a bug rather than a finding.

In [ ]:
!python scripts/run_stage5_features.py --simulate --n-samples 200
!python scripts/validate_stage6_null.py \
    --features data/interim/sim_features.parquet --endpoint OS

## 7. Report

Regenerates every figure and table from cached artefacts and assembles the
report in markdown, self-contained HTML and Word. Recomputes no training.

In [ ]:
!python scripts/run_report.py
!python scripts/run_manuscript.py

import pathlib
for p in sorted(pathlib.Path('reports').glob('*')):
    print(f'{p}  {p.stat().st_size/1e6:.1f} MB')

In [ ]:
from google.colab import files

DOWNLOAD = False
if DOWNLOAD:
    files.download('reports/analysis_report.html')
    files.download('reports/manuscript.docx')